# 2. CLAP Embedding and Indexing

Turn every downloaded clip into a searchable vector.

```
your sound
    │
    ▼
48 kHz waveform
    │
    ▼
630k + AudioSet CLAP (fusion)
    │
    ▼
512-D embedding
    │
    ▼
L2 normalize
    │
    ▼
FAISS
    │
    ├── sound → sound similarity
    └── text  → sound semantic search
```

We keep speech labels from notebook 1 in the metadata so you can filter later.

## 0. Imports and paths

In [9]:
import json
from pathlib import Path

import faiss
import laion_clap
import librosa
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

ROOT = Path("..").resolve()
AUDIO_DIR = ROOT / "data" / "audio"
LABELS_CSV = ROOT / "data" / "audio_speech_labels.csv"
INDEX_DIR = ROOT / "data_index"
INDEX_DIR.mkdir(exist_ok=True)

INDEX_PATH = INDEX_DIR / "clap_embeddings.faiss"
META_PATH = INDEX_DIR / "clap_metadata.json"

# CLAP always wants 48 kHz audio
SR = 48_000
CHUNK_SEC = 10          # long clips get split into 10s pieces, then averaged
EMB_DIM = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}")
print(f"audio dir={AUDIO_DIR} exists={AUDIO_DIR.is_dir()}")

device=cuda
audio dir=C:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\SaysUserContentRecommendation\data\audio exists=True


## 1. Load the CLAP model

One model, two encoders: audio → 512-D and text → 512-D, in the same space.
We use the **fusion** checkpoint (`630k-audioset-fusion-best.pt`) so longer audio is handled better.

PyTorch 2.6+ defaults `torch.load(weights_only=True)`, which breaks LAION-CLAP’s pickle-based checkpoints. We temporarily allow `weights_only=False` only while loading this trusted official weight file.

In [10]:
# PyTorch 2.6+ defaults weights_only=True; LAION-CLAP ckpts need False
_orig_torch_load = torch.load

def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)

torch.load = _torch_load_compat
try:
    clap = laion_clap.CLAP_Module(enable_fusion=True, device=DEVICE)
    clap.load_ckpt()  # downloads 630k-audioset-fusion-best.pt on first run
finally:
    torch.load = _orig_torch_load

clap.eval()
print("CLAP ready")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1954.64it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.patch_embed.mel_conv2d.weight 	 Loaded
audio_branch.patch_embed.mel_conv2d.bias 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.0.weight 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.0.bias 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.1.weight 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.1.bias 	 Loaded
audio_branch.patch_embed.fusion_model.local_att.3.weight 	 Loaded
audio_branc

## 2. Helper functions (the whole pipeline)

Four tiny steps — nothing more:

1. **load** audio at 48 kHz  
2. **chunk** long files into ≤10s pieces  
3. **embed** with CLAP and average chunks  
4. **normalize** so FAISS inner-product ≈ cosine similarity

In [11]:
def load_waveform(path: Path) -> np.ndarray:
    """Step 1: sound → 48 kHz waveform."""
    audio, _ = librosa.load(path, sr=SR, mono=True)
    return audio


def make_chunks(audio: np.ndarray, chunk_sec: int = CHUNK_SEC) -> list[np.ndarray]:
    """Step 2: split into non-overlapping ≤chunk_sec pieces."""
    n = chunk_sec * SR
    if len(audio) <= n:
        return [audio]
    return [audio[i:i + n] for i in range(0, len(audio), n)]


def embed_audio(path: Path) -> np.ndarray | None:
    """Steps 1–4: waveform → CLAP → mean pool → L2 normalize → 512-D vector."""
    try:
        chunks = make_chunks(load_waveform(path))
        # CLAP returns one 512-D vector per chunk
        emb = clap.get_audio_embedding_from_data(x=chunks, use_tensor=False)
        vec = np.mean(emb, axis=0).astype("float32")
        # L2 normalize so ||v|| = 1
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec /= norm
        return vec
    except Exception as e:
        print(f"skip {path.name}: {e}")
        return None


def embed_text(query: str) -> np.ndarray:
    """Text → same 512-D space as audio (already L2-normalized by CLAP)."""
    vec = clap.get_text_embedding([query], use_tensor=False)[0].astype("float32")
    return vec


print("helpers ready")

helpers ready


## 3. Collect files to embed

Match files in `data/audio/` to rows in `audio_speech_labels.csv`.

In [12]:
labels = pd.read_csv(LABELS_CSV)
labels["id"] = labels["id"].astype(str)
is_speech = dict(zip(labels["id"], labels["is_speech"]))

audio_files = sorted(
    p for p in AUDIO_DIR.iterdir()
    if p.is_file() and p.stem in is_speech
)

print(f"labels: {len(is_speech)}")
print(f"audio files to embed: {len(audio_files)}")

labels: 6374
audio files to embed: 6374


## 4. Embed everything and build FAISS

`IndexFlatIP` = exact inner product.  
Because vectors are L2-normalized, **inner product = cosine similarity**.

In [13]:
index = faiss.IndexFlatIP(EMB_DIM)
metadata = []   # row i in metadata ↔ vector i in the index
failed = []

for path in tqdm(audio_files, desc="CLAP embed"):
    vec = embed_audio(path)
    if vec is None:
        failed.append(path.name)
        continue

    index.add(vec.reshape(1, -1))
    metadata.append({
        "id": path.stem,
        "filename": path.name,
        "is_speech": bool(is_speech[path.stem]),
    })

print(f"indexed {index.ntotal} clips")
if failed:
    print(f"failed: {len(failed)}")

CLAP embed:   0%|          | 0/6374 [00:00<?, ?it/s]c:\Users\utkar\.conda\envs\conda-nlp312\Lib\site-packages\torchaudio\transforms\_transforms.py:590: UserWarning: Argument 'onesided' has been deprecated and has no influence on the behavior of this module.
  warnings.warn(
CLAP embed: 100%|██████████| 6374/6374 [44:13<00:00,  2.40it/s]  

indexed 6374 clips


## 5. Save index + metadata

In [14]:
faiss.write_index(index, str(INDEX_PATH))
META_PATH.write_text(json.dumps(metadata, indent=2))

print(f"saved {INDEX_PATH}")
print(f"saved {META_PATH}")

saved C:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\SaysUserContentRecommendation\data_index\clap_embeddings.faiss
saved C:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\SaysUserContentRecommendation\data_index\clap_metadata.json


## 6. Search demos

Reload is optional if you just built the index above — handy when you reopen the notebook later.

In [18]:
index = faiss.read_index(str(INDEX_PATH))
metadata = json.loads(META_PATH.read_text())


def search(query_vec: np.ndarray, k: int = 5):
    """Return top-k (score, metadata) pairs."""
    scores, ids = index.search(query_vec.reshape(1, -1).astype("float32"), k)
    return [
        (float(scores[0][i]), metadata[int(ids[0][i])])
        for i in range(k)
        if ids[0][i] >= 0
    ]


def show(results):
    for score, meta in results:
        print(f"  {score:.3f}  {meta['filename']}  speech={meta['is_speech']}")

### 6a. Sound → sound

Pick one clip, embed it, find the most similar clips in the index.

In [19]:
query_path = audio_files[0]
print(f"query: {query_path.name}")

q = embed_audio(query_path)
show(search(q, k=5))

query: 00057e4c-6542-4bed-be04-a308002dda40.mp3
  1.000  f5c39eae-a28e-4ed1-a682-f5451847d8bd.mp3  speech=True
  1.000  00057e4c-6542-4bed-be04-a308002dda40.mp3  speech=True
  0.906  7985e245-9612-44c2-858d-a164f807f7b1.mp3  speech=True
  0.890  99b8970b-81d4-4e02-b4de-1d465b3d36d1.mp3  speech=False
  0.888  505e3298-d1dd-42bd-9b83-f39bb6d3bf7b.mp3  speech=False


### 6b. Text → sound

Same index, different encoder. Type a description; CLAP finds matching audio.

In [20]:
query_text = "people talking in a noisy street"
print(f"query: {query_text!r}")

q = embed_text(query_text)
show(search(q, k=5))

query: 'people talking in a noisy street'
  0.458  6ad72c31-51be-4f6e-bcf4-8e39b0f80991.mp3  speech=False
  0.397  1cd88677-aa73-4b42-a6ad-fa99f7ecf175.mp3  speech=False
  0.343  44dd6765-0077-40af-9a03-f550ad3b8b31.mp3  speech=False
  0.336  37b8efc4-afe7-421b-92ab-d2f49124e9f3.mp3  speech=False
  0.322  d43e2315-1def-475e-87dd-a2c6c58de83d.mp3  speech=False
